In [ ]:
# Import necessary libraries and modules
import pandas as pd
import yaml
from pathlib import Path
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401 — registers PD metrics
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_roc, plot_gini, plot_ks_cdf_with_maxgap
from IPython.display import Image, display  

# Set constants for data loading and RAG configuration
DATA_DIR = Path('..') / 'data'
csv_path = DATA_DIR / 'sample.csv'
rag_path = Path('../configs/validation_rag.yaml')

# Load the dataset
print("Loading data...")
df = pd.read_csv(csv_path)

# Define year-specific subsets for development and validation
dev_year = 2019
val_year = 2020
year_col = 'score_year'

df_dev = df[df[year_col] == dev_year].copy()
df_val = df[df[year_col] == val_year].copy()

# Initialize the metrics service with desired metric IDs
metric_ids = ['auc_roc', 'gini', 'ks']
service = PDMetricsService(metric_ids)

# Set parameters for metrics calculation
params = {
    'y_col': 'default_flag',
    'p_col': 'pred_br',
    'score_col': 'score_pd',
    # 'period_col': 'QTR',  # Optional: for per-period metrics, include if applicable
    'band_col': 'BAND'
}

# Calculating metrics for development and validation datasets
print("Calculating metrics for dev set...")
dev_results = service.compute(df_dev, params)

print("Calculating metrics for val set...")
val_results = service.compute(df_val, params)

# Display and visualize the results
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

# Plot Gini (Gini is often visualized as a Lorenz curve)
gini_path = plot_gini(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='Validation Gini Coefficient',
    out_path=plot_dir / 'gini.png')
display(Image(filename=str(gini_path)))

# Plot ROC curve for AUC
roc_path = plot_roc(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='Validation ROC Curve',
    out_path=plot_dir / 'roc_curve.png')
display(Image(filename=str(roc_path)))

# Plot KS CDF
eks_path, _ = plot_ks_cdf_with_maxgap(
    y_true=df_val['default_flag'],
    y_score=df_val['pred_br'],
    title='Validation KS CDF',
    out_path=plot_dir / 'ks_cdf.png')
display(Image(filename=str(eks_path)))

# Optional: Load the RAG policy thresholds for further analysis
with open(rag_path) as f:
    rag_config = yaml.safe_load(f)

# Note: Further code can be added to compute the RAG status using the loaded thresholds